# AQI Card Debugging
This notebook inspects the AQI card source, reproduces the failure, and validates the fix.

In [ ]:
import json
import pathlib
import urllib.request

root = pathlib.Path(r"c:\Users\swags\OneDrive\Documents\GitHub\EnviroMonitor")

## Load AQI Card Source Code
Inspect the live AQI client and server route implementations to identify where the data is selected and rendered.

In [ ]:
client_path = root / 'public' / 'pages' / 'index.html'
server_path = root / 'server.js'
print('Client file:', client_path)
print('Server file:', server_path)

with client_path.open('r', encoding='utf-8') as f:
    client_source = f.read()
with server_path.open('r', encoding='utf-8') as f:
    server_source = f.read()

print('Client excerpt:')
print(client_source[client_source.find('async function fetchLiveAQI'):
                   client_source.find('function calculateUSAQI', client_source.find('async function fetchLiveAQI'))][:400])
print('\nServer excerpt:')
print(server_source[server_source.find("app.get('/api/live-aqi'"):
                   server_source.find('const PROFESSIONAL_EMAIL_REGEX', server_source.find("app.get('/api/live-aqi'"))][:500])

## Reproduce the AQI Card Failure
Run a minimal request against Open-Meteo and the local API route to compare returned payloads and identify invalid values.

In [ ]:
# Fetch Open-Meteo raw data for Delhi
url = 'https://air-quality-api.open-meteo.com/v1/air-quality?latitude=28.7041&longitude=77.1025&hourly=pm10,pm2_5,nitrogen_dioxide,sulphur_dioxide&timezone=auto'
with urllib.request.urlopen(url, timeout=30) as response:
    raw = response.read().decode('utf-8')

payload = json.loads(raw)
print('Open-Meteo keys:', list(payload.keys()))
print('Hourly keys:', list(payload.get('hourly', {}).keys()))
print('Time length:', len(payload['hourly']['time']))
print('Last time:', payload['hourly']['time'][-1])
print('Last pm2_5 value:', payload['hourly']['pm2_5'][-1])
print('Last pm10 value:', payload['hourly']['pm10'][-1])
print('Last no2 value:', payload['hourly']['nitrogen_dioxide'][-1])
print('Last so2 value:', payload['hourly']['sulphur_dioxide'][-1])
print('Last valid pm2_5 index:', next((i for i in range(len(payload['hourly']['time'])-1, -1, -1) if payload['hourly']['pm2_5'][i] is not None), None))

## Inspect Data Inputs and API Response
Verify why the last hourly index is invalid and how the rendered AQI values are derived from the payload.

In [ ]:
# Determine the last valid data index across all pollutants
hourly = payload['hourly']
key_names = ['pm2_5', 'pm10', 'nitrogen_dioxide', 'sulphur_dioxide']
last_index = next((i for i in range(len(hourly['time'])-1, -1, -1)
                   if any(hourly.get(key)[i] is not None for key in key_names)), None)
print('Last valid index:', last_index)
print('Last valid time:', hourly['time'][last_index])
print({key: hourly.get(key)[last_index] for key in key_names})

## Debug Rendering and State Management
Ensure the frontend uses the same valid index selection strategy and that the fallback path calculates AQI correctly.

In [ ]:
print('AQI calculation function exists in source:', 'function calculateUSAQI' in client_source)
print('Fallback code excerpt:')
fallback_start = client_source.find('const altUrl')
print(client_source[fallback_start:fallback_start+500])

## Apply Fix and Test with Sample AQI Data
The fix should use the latest available non-null index for hourly pollutant measurements and compute AQI from the selected values.

In [ ]:
# Validate the logic that should be used by the fix
if last_index is not None:
    sample_components = {key: hourly[key][last_index] for key in key_names}
    print('Sample components at last valid index:', sample_components)
else:
    print('No valid sample index found.')

# Confirm the corrected calculation would produce a usable AQI value
def calculate_aqi(components):
    pm25 = components['pm2_5'] or 0
    if pm25 <= 12: return pm25 * (50 / 12)
    if pm25 <= 35.4: return 50 + ((pm25 - 12) * (100 - 50) / (35.4 - 12))
    if pm25 <= 55.4: return 100 + ((pm25 - 35.4) * (150 - 100) / (55.4 - 35.4))
    if pm25 <= 150.4: return 150 + ((pm25 - 55.4) * (200 - 150) / (150.4 - 55.4))
    return 200 + ((pm25 - 150.4) * (300 - 200) / (250.4 - 150.4))

print('Calculated AQI:', calculate_aqi(sample_components))